# VIVA AI — Soil Classifier Training v2 (Colab, free GPU tier)

**This notebook was rebuilt after the v1 dataset link died (HTTP 403) and
the original conversion step was found to break Colab's Python
environment.** Two real fixes are built in this time:

1. **Auto-discovery instead of guessing.** Rather than hardcoding exact
   Kaggle folder names sight-unseen (what broke last time), this notebook
   downloads the dataset, walks every folder, counts real image files in
   each, and prints a full report — then fuzzy-matches folder names
   against the soil classes we want. If the dataset's actual structure
   doesn't match what's expected, you'll see a clear report telling you
   exactly what *is* there, instead of a cryptic crash deep inside Keras.
2. **Training and TF.js conversion are split into two separate runtime
   sessions.** Installing the `tensorflowjs` pip package in the same
   Python process that already has TensorFlow loaded and a trained model
   in memory is a well-known, frequently-reported way to corrupt the
   Colab runtime — pip's dependency resolver fights the GPU runtime's
   pre-installed TensorFlow build, sometimes silently breaking things
   rather than failing loudly. This notebook saves the trained model to
   disk, has you do a **Runtime → Restart session** (this keeps your
   files on disk, just clears Python state — it takes 10 seconds, not a
   new VM), and converts to TF.js in that clean session instead.

**Before running:** Runtime → Change runtime type → T4 GPU (free tier).


## 1. Get a free Kaggle API token

kaggle.com → your profile → Account → "Create New API Token" → downloads `kaggle.json`. Upload it when prompted below.

In [ ]:
from google.colab import files
print("Upload your kaggle.json (from Kaggle > Account > Create New API Token)")
uploaded = files.upload()


In [ ]:
import os
os.makedirs("/root/.kaggle", exist_ok=True)
os.system("cp kaggle.json /root/.kaggle/kaggle.json")
os.system("chmod 600 /root/.kaggle/kaggle.json")
print("Kaggle credentials installed.")


## 2. Download the soil dataset

Tries a primary dataset; if it's unavailable (renamed, removed, or
access-restricted — this exact failure is what broke the previous
version of this notebook), automatically tries a backup. Either way, the
next section discovers the real folder structure rather than assuming it.

In [ ]:
!pip install -q kaggle

import subprocess

DATASET_CANDIDATES = [
    "ai4a-lab/comprehensive-soil-classification-datasets",
    "jhislainematchouath/soil-types-dataset",
]

download_ok = False
for slug in DATASET_CANDIDATES:
    print(f"Trying dataset: {slug} ...")
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", slug, "-p", "/content/soil_raw", "--unzip"],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(f"SUCCESS downloading {slug}")
        download_ok = True
        break
    else:
        print(f"FAILED ({slug}): {result.stderr.strip()[-400:]}")
        print("Trying next candidate, if any...\n")

if not download_ok:
    raise RuntimeError(
        "Could not download any candidate dataset. Go to kaggle.com, search "
        "'soil classification', pick a dataset with at least 3-4 distinct soil "
        "type folders, copy its slug (the 'owner/dataset-name' part of the URL), "
        "and add it to DATASET_CANDIDATES above, then re-run this cell."
    )


## 3. Auto-discover the real folder structure (don't assume it)

This is the key fix from v1: instead of hardcoding folder names we
haven't actually seen, this walks the entire downloaded dataset, finds
every folder that directly contains image files, counts them, and prints
a full report. **Read this report before continuing** — it tells you
exactly what classes are really available in whatever got downloaded.

In [ ]:
import os

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def find_image_folders(root):
    """Returns {folder_path: image_count} for every folder that directly
    contains at least one image file, anywhere under root, at any depth."""
    found = {}
    for dirpath, _, filenames in os.walk(root):
        images = [f for f in filenames if f.lower().endswith(IMAGE_EXTENSIONS)]
        if images:
            found[dirpath] = len(images)
    return found

image_folders = find_image_folders("/content/soil_raw")

print(f"Found {len(image_folders)} folders containing images:\n")
for path, count in sorted(image_folders.items(), key=lambda x: -x[1]):
    label = os.path.basename(path.rstrip("/"))
    print(f"  {count:5d} images  —  {label:30s}  ({path})")

assert image_folders, (
    "No image folders found at all under /content/soil_raw. The dataset "
    "download likely produced an unexpected archive structure — run "
    "'!find /content/soil_raw -maxdepth 4' in a new cell to inspect it manually."
)


## 4. Fuzzy-match discovered folders to our 4 target soil classes

Matches folder names against keyword groups for our target classes
(alluvial, black, red, and a flexible 4th slot that accepts clay,
laterite, or yellow soil — whichever your downloaded dataset actually
has, since that varies by source). **If this cell's printed mapping
looks wrong, edit `MATCHED` by hand before continuing** — that's exactly
what it's designed to make easy, instead of failing silently.

In [ ]:
KEYWORD_GROUPS = {
    "alluvial": ["alluvial"],
    "black":    ["black", "regur"],
    "red":      ["red"],
    "fourth":   ["clay", "laterite", "yellow"],  # accepts whichever exists
}

def normalize(name):
    return name.lower().replace("soil", "").replace("_", " ").replace("-", " ").strip()

MATCHED = {}
for path, count in image_folders.items():
    label = normalize(os.path.basename(path.rstrip("/")))
    for canonical, keywords in KEYWORD_GROUPS.items():
        if canonical in MATCHED:
            continue  # already matched this class to a (presumably larger) folder
        if any(kw in label for kw in keywords):
            MATCHED[canonical] = (path, count)

print("Matched classes:")
for canonical, (path, count) in MATCHED.items():
    print(f"  {canonical:10s} -> {os.path.basename(path)}  ({count} images)")

missing = set(KEYWORD_GROUPS) - set(MATCHED)
if missing:
    print(f"\nCould NOT auto-match: {missing}")
    print("Look at the folder list printed in the previous cell and manually")
    print("add entries to MATCHED, e.g.: MATCHED['fourth'] = ('/content/soil_raw/Yellow_Soil', 120)")

assert len(MATCHED) >= 3, (
    "Fewer than 3 usable soil classes were matched — that's too few for a "
    "meaningful classifier. Check the folder report above and either pick a "
    "different dataset or extend KEYWORD_GROUPS to match what's really there."
)
CANONICAL_CLASSES = sorted(MATCHED.keys())  # fixed, deterministic order
print(f"\nFinal class list (in this fixed order): {CANONICAL_CLASSES}")


## 5. Copy into a clean, consistently-named dataset folder

In [ ]:
import shutil, glob

DST_ROOT = "/content/dataset"
os.makedirs(DST_ROOT, exist_ok=True)

for canonical in CANONICAL_CLASSES:
    src_path, _ = MATCHED[canonical]
    dst_dir = os.path.join(DST_ROOT, canonical)
    os.makedirs(dst_dir, exist_ok=True)
    copied = 0
    for f in os.listdir(src_path):
        if f.lower().endswith(IMAGE_EXTENSIONS):
            shutil.copy(os.path.join(src_path, f), dst_dir)
            copied += 1
    print(f"{canonical}: copied {copied} images")


## 6. Train MobileNetV2 transfer-learning model

Deliberately does NOT `pip install` or upgrade TensorFlow here — uses whatever Colab's GPU runtime already has pre-installed. Fighting that pre-installed version is exactly what caused the environment corruption this notebook is designed to avoid.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TensorFlow version (Colab's pre-installed build, untouched):", tf.__version__)

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 15
NUM_CLASSES = len(CANONICAL_CLASSES)

train_gen = ImageDataGenerator(
    rescale=1.0 / 255, rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, horizontal_flip=True,
    brightness_range=[0.8, 1.2], validation_split=0.2,
)

train_data = train_gen.flow_from_directory(
    DST_ROOT, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="training", classes=CANONICAL_CLASSES,
)
val_data = train_gen.flow_from_directory(
    DST_ROOT, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="validation", classes=CANONICAL_CLASSES,
)

print("\nClass index mapping (this fixed order matters for the app's SOIL_LABELS):")
print(train_data.class_indices)


In [ ]:
base_model = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
history = model.fit(train_data, validation_data=val_data, epochs=EPOCHS)


## 7. Check validation accuracy, fine-tune if needed

In [ ]:
val_loss, val_acc = model.evaluate(val_data)
print(f"Validation accuracy: {val_acc:.2%}")

if val_acc < 0.70:
    print("Fine-tuning: unfreezing the last 20 layers of MobileNetV2...")
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                   loss="categorical_crossentropy", metrics=["accuracy"])
    model.fit(train_data, validation_data=val_data, epochs=5)
    val_loss, val_acc = model.evaluate(val_data)
    print(f"Validation accuracy after fine-tuning: {val_acc:.2%}")


## 8. Save the model to disk — STOP after this cell

Do **not** run the TF.js conversion in this same session. Save here,
then follow the restart instructions in the next markdown cell.

In [ ]:
model.save("/content/soil_model_keras.h5")
print("Saved to /content/soil_model_keras.h5")
print("\nFinal class order (write this down, you'll need it after restart):")
print(CANONICAL_CLASSES)

with open("/content/soil_class_order.json", "w") as f:
    import json as _json
    _json.dump(CANONICAL_CLASSES, f)
print("Also saved to /content/soil_class_order.json so it survives the restart.")


## 9. ⚠️ Restart the runtime now, then continue below

**Runtime menu (top) → Restart session.** This takes about 10 seconds
and clears Python's memory, but your files in `/content/` (including the
model you just saved) are untouched — this is the Colab VM's disk, not
your Python variables.

After it restarts, **do not re-run cells 1-8.** Just run the cells below
this point, starting with the next one. This keeps the `tensorflowjs`
pip install completely isolated from the TensorFlow build that did the
training — that isolation is what avoids the environment corruption that
broke the previous version of this notebook.

In [ ]:
# Run this AFTER Runtime > Restart session — this cell intentionally does
# NOT import tensorflow for training; only what's needed to convert the
# already-saved model file on disk.
import json as _json

with open("/content/soil_class_order.json") as f:
    CANONICAL_CLASSES = _json.load(f)
print("Reloaded class order:", CANONICAL_CLASSES)

import os
assert os.path.exists("/content/soil_model_keras.h5"), (
    "soil_model_keras.h5 not found — if you disconnected the whole runtime "
    "(not just 'restart session'), the VM's disk was wiped and you'll need "
    "to re-run training from the top. 'Restart session' (not 'Disconnect and "
    "delete runtime' or a new Colab tab) keeps files; that distinction is "
    "exactly why this step exists."
)
print("Found saved model on disk, proceeding to convert.")


## 10. Install tensorflowjs fresh, in this clean session, and convert

In [ ]:
!pip install -q tensorflowjs

# Using the command-line converter (not the Python API) on the .h5 file
# directly — this avoids ever having a live TensorFlow training session
# and the tensorflowjs package's own bundled dependencies in the same
# process at the same time, which is the actual mechanism behind the
# environment corruption this notebook works around.
!tensorflowjs_converter --input_format=keras \
    /content/soil_model_keras.h5 \
    /content/soil_model_tfjs

print("Conversion complete. Files:")
!ls -la /content/soil_model_tfjs


## 11. Download and install into the app

Copy every file from the downloaded zip into
`public/models/soil_model/` in the VIVA AI project, replacing the
placeholder README there. No app code changes needed —
`src/lib/mockModel.ts` detects and loads it automatically.

**Class order:** this notebook fixed `CANONICAL_CLASSES` to a sorted,
deterministic list at step 4, printed again at step 8, and reloaded
fresh after the restart — there is no ambiguity to resolve by hand this
time. Open `src/lib/mockModel.ts`, find `SOIL_LABELS`, and set it to the
Tamil translations of `CANONICAL_CLASSES` **in that exact printed
order** — for example, if it printed `['alluvial', 'black', 'fourth',
'red']`, then `SOIL_LABELS` should be `["வண்டல் மண்", "கருமண்",
"<translation of whatever 'fourth' matched>", "செம்மண்"]`, in that
order, not alphabetical or any other order.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/soil_model_tfjs", "zip", "/content/soil_model_tfjs")
files.download("/content/soil_model_tfjs.zip")
